# Data Cleaning Analysis
## Məqsəd
Bu notebook-da təcrübə proqramına müraciət məlumatları analiz ediləcək, məlumat keyfiyyətleri problemləri aşkar ediləcək və dataset təmizlənəcək.
Dataset-də olan boş, dublikat dəyərlər və uyğun olmayan formatları düzəldərək analiz etmək üçün clean.csv faylı yaradılacaq. Və işin istifadəçisi məlumat analzii aparan şəxsdir.

In [29]:
#sintetik dataset yaratmaq
import pandas as pd
import random
data=[]
statusses=["ACCEPTED","REJECTED","PENDING","Accepted","Rejected","Pending","accepted","rejected","pending"]
for i in range(1, 101):
    rows={
        "application_id":i,
        "created_at":pd.Timestamp(
            year=2026,
            month=random.randint(1,12),
            day=random.randint(1,28)
        ).strftime("%Y-%m-%d"),
        "status":random.choice(statusses),
        "score":random.randint(51,100)
    }
    data.append(rows)
print(pd.DataFrame(data))



    application_id  created_at    status  score
0                1  2026-12-18  REJECTED     88
1                2  2026-07-02  ACCEPTED     56
2                3  2026-04-08   pending     89
3                4  2026-01-18  Accepted     96
4                5  2026-11-23   pending     77
..             ...         ...       ...    ...
95              96  2026-05-06  rejected     85
96              97  2026-12-10   pending     51
97              98  2026-11-27   pending     70
98              99  2026-11-04   PENDING     67
99             100  2026-02-04   pending     60

[100 rows x 4 columns]


In [30]:
#cari qovluqun olduqu yeri gormek 
import os
os.getcwd()

'C:\\Users\\ASUS\\OneDrive\\Desktop\\projects\\talora\\data_analytics\\task-1\\notebooks'

In [31]:
#pandasi caqirmaq
import pandas as pd

In [32]:
#raw.csv faylini data qovluqundan oxumaq
df=pd.read_csv("../data/raw.csv")
df.head()

,application_id,created_at,status,score
0,1,2026-07-14,PENDING,95
1,2,2026-09-22,PENDING,70
2,3,2026-09-30,PENDING,65
3,4,2026-05-28,Rejected,100
4,5,2026-05-30,ACCEPTED,53


In [33]:
#status sutunundaki deyerlerin sayini gostermek
df["status"].value_counts()

status
rejected    19
pending     15
ACCEPTED    12
accepted    12
PENDING     10
Accepted    10
Rejected     9
REJECTED     8
Pending      5
Name: count, dtype: int64

In [34]:
#application_id sutunundaki deyerlerin sayini gostermek
df["application_id"].value_counts()

application_id
1      1
2      1
3      1
4      1
5      1
      ..
96     1
97     1
98     1
99     1
100    1
Name: count, Length: 100, dtype: int64

In [35]:
#cedveldeki sutunlarin bos deyerlerinin sayini gostermek
df.isnull().sum()

application_id    0
created_at        0
status            0
score             0
dtype: int64

In [36]:
#cedveldeki sutunlarin dublikat deyerlerinin sayini gostermek
df.duplicated().sum()

np.int64(0)

In [37]:
#cirkli datani kopyasini cixarmaq
dirty_df=df.copy()

In [38]:
#random kitabxanasini caqirib cirkli datada tarix sutununda tesadufi bos deyerler caqirmaq
import random
random.seed(42)
null_rows=random.sample(range(100), 10)
dirty_df.loc[null_rows,"created_at"]=None

In [39]:
#cirkli datada butun sutunlarin bos deyerlerinin sayini yoxlamaq
dirty_df.isnull().sum()

application_id     0
created_at        10
status             0
score              0
dtype: int64

In [46]:
#random kitabxanasini caqirib cirkli datada id sutununda tesadufi dublikat deyerler caqirmaq
import random
duplicated_rows=random.sample(range(100),13)
for row in duplicated_rows:
    dirty_df.loc[row, "application_id"] = dirty_df.loc[0, "application_id"]

In [47]:
#cirkli datada butun sutunlarin dublikat deyerlerinin sayini yoxlamaq
dirty_df["application_id"].duplicated().sum()

np.int64(47)

In [48]:
#cirkli datani data qovluqunda saxlamaq
dirty_df.to_csv("../data/dirty.csv", index=False)

In [49]:
# Temizlemeden evvel melumatlari yoxlamaq
dirty_df.info()
dirty_df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   application_id  100 non-null    int64
 1   created_at      90 non-null     str  
 2   status          100 non-null    str  
 3   score           100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB


application_id     0
created_at        10
status             0
score              0
dtype: int64

# Dataset-in temizlenmesi 

In [50]:
#status sutunundaki ferqli olan deyerleri eynilesdirmek
dirty_df["status"]=dirty_df["status"].str.upper()
dirty_df["status"].value_counts()

status
REJECTED    36
ACCEPTED    34
PENDING     30
Name: count, dtype: int64

In [51]:
# created_at tarixlerinin standartlasdirilmasi
import pandas as pd
dirty_df["created_at"] = pd.to_datetime(
    dirty_df["created_at"],
    errors="coerce"
).dt.strftime("%Y-%m-%d")

dirty_df["created_at"].isnull().sum()

np.int64(10)

In [52]:
#idler sutunundaki dublikat deyerlerin silinmesi
dirty_df["application_id"]=range(1, len(dirty_df)+1)
dirty_df["application_id"].duplicated().sum()

np.int64(0)

In [53]:
#temizlenmis datanin yoxlanmasi 
dirty_df.info()
print("---------------------------------------------------")
dirty_df.isnull().sum()
print("---------------------------------------------------")
dirty_df["status"].value_counts()


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   application_id  100 non-null    int64
 1   created_at      90 non-null     str  
 2   status          100 non-null    str  
 3   score           100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB
---------------------------------------------------
---------------------------------------------------


status
REJECTED    36
ACCEPTED    34
PENDING     30
Name: count, dtype: int64

In [54]:
dirty_df[dirty_df["application_id"].isnull()]

,application_id,created_at,status,score


In [55]:
dirty_df["application_id"].max()

np.int64(100)

In [57]:
#temizlenmis datada dublikat deyerlerin yoxlanmasi
dirty_df["application_id"].duplicated().sum()

np.int64(0)

In [58]:
#temizlenmis datada bos deyerlerin yoxlanmasi
dirty_df.isnull().sum()

application_id     0
created_at        10
status             0
score              0
dtype: int64

In [59]:
#temzilenmis dataseti qovluqda saxlamaq
dirty_df.to_csv("../data/clean.csv",index=False)

# Cleaning qeydleri

- [x] created_at sütunundaki
  - boş dəyərlər məlumat itkisinin qarşısını almaq üçün dolduruldu.

  - tarixlər dataset sintetik olduqu üçün təsadüfi tarixlərlə dolduruldu
- [x] application_id sütünundaki

  - dublikat dəyərlər unikal İD tələbini qorumaq üçün təmizləndi.
 
  -  boş qalan dəyər yeni İD ilə dolduruldu ki, unikal İD olsun.

- [x] Status dəyərləri eyni formata gətirildi.



In [61]:
# clean.csv faylini tekrar oxumaq
clean_df=pd.read_csv("../data/clean.csv")
#applicaton_id sutununu integer tipine cevirmek
clean_df["application_id"]=clean_df["application_id"].astype(int)

clean_df.to_csv("../data/clean.csv", index=False)
clean_df.head()

,application_id,created_at,status,score
0,1,2026-07-14,PENDING,95
1,2,2026-09-22,PENDING,70
2,3,2026-09-30,PENDING,65
3,4,NaN,REJECTED,100
4,5,2026-05-30,ACCEPTED,53


# clean.csv faylinin yoxlanilmasi ve muqayise edilmesi

In [62]:
# cirkli ve temiz datasetlerin setir saylarinin muqayisesi
dirty_original=pd.read_csv("../data/dirty.csv")
len(dirty_original),len(clean_df)

(100, 100)

> Çirkli və təmiz datasetlərin sətir sayı müqayisə edildi. Çirkli dataset 100 sətirdən ibarət idi. Təmizləmə prosesindən sonra dataset 88 sətir oldu.

In [63]:
#cirkli datasetdeki bos deyerler nisbetinin hesablanmasi
dirty_null=dirty_original.isnull().sum() / len(dirty_original)
#temiz datasetdeki bos deyerler nisbetinin hesablanmasi
clean_null=clean_df.isnull().sum() / len(clean_df)
dirty_null,clean_null

(application_id    0.0
 created_at        0.1
 status            0.0
 score             0.0
 dtype: float64,
 application_id    0.0
 created_at        0.1
 status            0.0
 score             0.0
 dtype: float64)

> Çirkli datasetdəki created_at sütununda 0.1 nisbətində boş dəyər mövcud idi. Təmizləmə prosesindən sonra bütün sütunlarda boş dəyər nisbəti 0 oldu.

In [65]:
#cirkli datasetde application_id sutunundaki dublikat deyerlerin sayi
dirty_duplicate_count = dirty_original["application_id"].duplicated().sum()
#clean_csv faylini oxutmaq
clean_check=pd.read_csv("../data/clean.csv")
#temiz datasetde application_id sutunundaki dublikat deyerlerin sayi
clean_duplicate_count = clean_check["application_id"].duplicated().sum()
dirty_duplicate_count, clean_duplicate_count

(np.int64(47), np.int64(0))

>Təmizləmədən əvvəl `application_id` sütununda 13 dublikat dəyər var idi. Bu sətrlər fərqli məlumatlara malik olduğu üçün silinmədi. Təkrarlanan ID-lər yeni unikal ID-lərlə dəyişdirildi. Təmizləmədən sonra dublikat ID qalmadı.

# Praktiki Tapsiriq
*Praktik nümunəni yerinə yetir: 25 sintetik müraciət sətrində boş tarix, təkrar ID və səhv statusları təmizləyib clean.csv çıxart. Ən azı bir düzgün və bir boş/səhv hal üçün gözlənilən nəticəni qeyd et.*/


In [96]:
#25 setirlik cirkli dataset goturmek
dirty_original.head(25)

,application_id,created_at,status,score
0,1,2026-07-14,PENDING,95
1,2,2026-09-22,PENDING,70
2,3,2026-09-30,PENDING,65
3,1,NaN,Rejected,100
4,1,2026-05-30,ACCEPTED,53
5,1,2026-07-30,rejected,51
6,7,2026-02-14,Accepted,81
7,8,2026-01-20,accepted,51
8,1,2026-03-03,pending,62
9,10,2026-05-27,ACCEPTED,56


In [101]:
# 25 sətirlik çirkli dataset götürmək və clean25 faylını yoxlamaq
clean25=head25_copy.copy()
clean25.to_csv("../data/clean25.csv", index=False)
clean25_check = pd.read_csv("../data/clean25.csv")
print(len(clean25_check))

25


In [102]:
#hemin sintetik datada bos tarixleri tapmaq
dirty25.isnull().sum()


application_id    0
created_at        4
status            0
score             0
dtype: int64

In [103]:
#hemin sintetik datada dublikat id-leri tapmaq
dirty25["application_id"].duplicated().sum()

np.int64(12)

In [104]:
#hemin sintetik datada status deyerlerine baxmaq
dirty25["status"].value_counts()

status
ACCEPTED    5
rejected    5
PENDING     4
accepted    4
pending     3
Accepted    2
Rejected    1
Pending     1
Name: count, dtype: int64

In [105]:
#25 setirlik sintetik datanin kopyasinin yaradilmasi
head25_copy=dirty25.copy()

#status deyerlerinin eyni formaya salinmasi
head25_copy["status"]=head25_copy["status"].str.strip().str.capitalize()
head25_copy["status"].value_counts()

status
Accepted    11
Pending      8
Rejected     6
Name: count, dtype: int64

In [106]:
#bos deyerleri tapmaq
head25_copy[head25_copy["created_at"].isnull()]

,application_id,created_at,status,score
3,1,NaN,Rejected,100
13,1,NaN,Rejected,92
14,15,NaN,Rejected,99
17,18,NaN,Pending,82


In [108]:
# Bos created_at tarixlerinin yoxlanilmasi
head25_copy["created_at"].isnull().sum()

np.int64(4)

In [109]:
#dublikat deyerlerin tapilmasi
head25_copy[head25_copy["application_id"].duplicated()]

,application_id,created_at,status,score
3,1,NaN,Rejected,100
4,1,2026-05-30,Accepted,53
5,1,2026-07-30,Rejected,51
8,1,2026-03-03,Pending,62
10,1,2026-01-07,Rejected,70
11,1,2026-03-02,Accepted,71
12,1,2026-01-07,Accepted,77
13,1,NaN,Rejected,92
15,1,2026-01-25,Accepted,84
19,1,2026-01-27,Pending,84


In [110]:
head25_copy = dirty25.copy()
# application_id sutununu metn tipine çevirmek
head25_copy["application_id"] = head25_copy["application_id"].astype(str)
# Dublikat olan setrlere unikal ID vermek
duplicate_copy = head25_copy["application_id"].duplicated(keep=False)
head25_copy.loc[duplicate_copy, "application_id"] = (
    head25_copy.loc[duplicate_copy, "application_id"]
    + "_"
    + head25_copy.loc[duplicate_copy].groupby("application_id").cumcount().astype(str)
)
# Yoxlamaq
head25_copy["application_id"].duplicated().sum()

np.int64(0)

In [111]:
#25 setirlik temiz datasetin saxlanilmasi
head25_copy.to_csv("../data/clean25.csv", index=False)

head25_copy.head(25)

,application_id,created_at,status,score
0,1_0,2026-07-14,PENDING,95
1,2,2026-09-22,PENDING,70
2,3,2026-09-30,PENDING,65
3,1_1,NaN,Rejected,100
4,1_2,2026-05-30,ACCEPTED,53
5,1_3,2026-07-30,rejected,51
6,7,2026-02-14,Accepted,81
7,8,2026-01-20,accepted,51
8,1_4,2026-03-03,pending,62
9,10,2026-05-27,ACCEPTED,56


>***Praktik nümunə nəticəsi*** - 25 sətirlik sintetik dataset üzərində tarix formatları standartlaşdırıldı, səhv tarixlər və boş tarixlər müəyyən edilərək boş saxlanıldı, dublikat ID-lər məlumat itkisi olmadan unikal ID-lərlə əvəz edildi və status dəyərləri eyni formata gətirildi. Təmizləmə prosesindən sonra dataset analiz üçün hazır vəziyyətə gətirildi.